# 02 — Eksperimen Model: Klasifikasi Volatilitas USD/IDR

Notebook ini berisi kode eksperimen model yang **sama persis** dengan skrip di `src/`
(disalin tanpa perubahan, hanya dipecah per blok top-level):

1. `dataset_split.py` — split kronologis 70/15/15, label `volatility_class` bebas bocor, grid tuning XGBoost
2. `train_baseline_ts.py` — baseline time-series: naive persistence & XGBoost harga saja
3. `train_tfidf.py` — teks saja: TF-IDF + XGBoost dan Sentiment Lexicon (Loughran-McDonald) + XGBoost
4. `train_combined.py` — model gabungan: harga + TF-IDF(SVD) + LM, ablasi, dan uji McNemar

Prasyarat: `data/processed/aligned_daily.csv` sudah ada (hasil `preprocess_kurs.py` →
`preprocess_news_tfidf.py` → `sentiment_lexicon.py` → `align.py`).
Notebook dijalankan dari folder `notebook/`. Sama seperti skrip `.py`, menjalankan ulang
notebook ini akan menyimpan model ke `models/` dan laporan ke `logs/`.

## 0. Setup

In [1]:
import sys
from pathlib import Path

# kode eksperimen ada di src/ -- tambahkan ke path supaya `import config` dan
# `from dataset_split import ...` jalan persis seperti di file .py
SRC_DIR = (Path.cwd().parent / "src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print("src dir:", SRC_DIR)

src dir: /Users/macbook/Documents/Sem5/NLP/Nyimut-NLP-26/src


## 1. `dataset_split.py` — Split data & label volatility_class

In [2]:
"""
dataset_split.py
=================
Split kronologis 70/15/15 (train/valid/test) + pelabelan volatility_class
BEBAS BOCOR -- dipakai bareng oleh SEMUA script training (train_tfidf.py,
train_baseline_ts.py, train_combined.py) supaya definisi split dan definisi
kelas volatility SAMA PERSIS di semua model, dan hasilnya bisa dibandingkan
apples-to-apples.

KENAPA AMBANG BATAS low/medium/high DI-FIT DARI TRAIN SAJA:
--------------------------------------------------------------------------
realized_vol (rolling std log_return, dihitung di preprocess_kurs.py) per
baris sendiri sudah aman -- cuma pakai histori KE BELAKANG. Tapi kalau
ambang batas (cutoff) low/medium/high dihitung pakai qcut dari SELURUH data
(termasuk valid & test), ambang batasnya "tahu" sebaran volatility masa
depan -- itu bocor, walau realized_vol sendiri tidak.

Analoginya sama persis kayak TfidfVectorizer: fit_transform di train,
transform DOANG (pakai vocab yang sudah dipelajari) di test. Di sini:
qcut (fit ambang batas) cuma di 70% data tertua, pd.cut (terapkan ambang
batas yang sama, tanpa dihitung ulang) ke train+valid+test sekaligus.

KENAPA BINNING DILAKUKAN SEBELUM DIPOTONG jadi train/valid/test:
--------------------------------------------------------------------------
lag1_volatility_class butuh label hari SEBELUMNYA. Kalau train/valid/test
dilabeli terpisah-pisah, baris PERTAMA di valid kehilangan lag1 dari baris
TERAKHIR di train (deretnya jadi bolong tepat di titik sambungan). Makanya:
label dulu SELURUH deret pakai ambang batas yang sudah di-fit, baru dipotong.
"""

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

import config as C

TRAIN_FRAC = 0.70
VALID_FRAC = 0.15
# sisa 0.15 -> test

VOLATILITY_LABELS = ["low", "medium", "high"]

# fitur harga yang SAMA dipakai XGBoost tanpa NLP (train_baseline_ts.py) dan
# XGBoost + TF-IDF + LM (train_combined.py) -- supaya selisih akurasinya murni
# efek fitur NLP, bukan beda fitur harga.
#
# KENAPA ADA known4_* DAN vol_if_*:
# target realized_vol(t) = std(r[t-4..t]) -- 4 dari 5 return di window itu
# SUDAH diketahui sebelum hari t, cuma r[t] yang belum. Makanya fitur yang
# paling informatif adalah ringkasan 4 return itu (bukan lag1_realized_vol,
# yang malah membawa r[t-5] -- return yang sudah keluar dari window target).
# vol_if_r0 = realized_vol(t) kalau r[t] = 0; vol_if_rtyp_* = kalau |r[t]|
# sebesar median |r| historis (naik / turun). Semua cuma pakai data < t.
PRICE_FEATURES = [
    "lag1_log_return", "lag2_log_return", "lag3_log_return", "lag4_log_return",
    "known4_mean", "known4_std", "vol_if_r0", "vol_if_rtyp_pos", "vol_if_rtyp_neg",
    "lag1_volatility_class_enc",
]

# fitur sentimen Loughran-McDonald per hari -- SAMA dipakai train_tfidf.py &
# train_combined.py. Pakai RATA-RATA PER ARTIKEL (bukan jumlah): jumlah artikel
# per hari berubah antar periode (rata2 3,9/hari di train vs 8,0 di valid, diduga
# karena cakupan scraping), jadi lm_*_sum ikut membesar walau nada beritanya
# sama. Rata-rata cuma mengukur NADA -- sejalan dgn Keputusan 18 (normalisasi
# volume berita). Lihat docs/keputusan.md.
LM_FEATURES = ["mean_lm_polarity", "lm_positive_mean", "lm_negative_mean"]

# grid kecil buat tuning -- dipilih berdasarkan akurasi VALID, dites ke test
# cuma SEKALI pakai kombinasi terbaik (bukan pilih2 sambil intip test).
# Dipakai SEMUA model XGBoost supaya masing-masing dapat kesempatan tuning yang sama.
XGB_GRID = [
    {"n_estimators": n, "max_depth": d, "learning_rate": lr}
    for n in (100, 200, 400)
    for d in (2, 3, 4)
    for lr in (0.03, 0.05, 0.1)
]

In [3]:
def add_price_features(df):
    """Fitur harga turunan log_return -- df HARUS deret kalender hari kerja lengkap & urut."""
    r = df["log_return"]
    df["lag4_log_return"] = r.shift(4)
    known = pd.concat([r.shift(i) for i in (1, 2, 3, 4)], axis=1)
    df["known4_mean"] = known.mean(axis=1, skipna=False)
    df["known4_std"] = known.std(axis=1, skipna=False)

    def vol_if(r_t):
        return np.std(np.column_stack([known.to_numpy(), r_t]), axis=1, ddof=1)

    typical = r.abs().expanding().median().shift(1).to_numpy()
    df["vol_if_r0"] = vol_if(np.zeros(len(df)))
    df["vol_if_rtyp_pos"] = vol_if(typical)
    df["vol_if_rtyp_neg"] = vol_if(-typical)
    return df

In [4]:
def tune_xgb(X_train, y_train_enc, X_valid, y_valid, X_test, encoder, log):
    """
    Coba semua kombinasi di XGB_GRID, PILIH berdasarkan akurasi VALID --
    test SAMA SEKALI tidak dilihat selama memilih. Kombinasi terbaik baru
    dites ke test SEKALI di akhir. Ini gunanya split 70/15/15 dibanding
    80/20: ada ruang buat coba-coba tanpa mengintip test.
    """
    best_acc, best_params, best_clf = -1, None, None
    for params in XGB_GRID:
        clf = XGBClassifier(**params, eval_metric="mlogloss", random_state=42)
        clf.fit(X_train, y_train_enc)
        pred_valid = encoder.inverse_transform(clf.predict(X_valid))
        acc = accuracy_score(y_valid, pred_valid)
        if acc > best_acc:
            best_acc, best_params, best_clf = acc, params, clf

    log(f"Hyperparameter terbaik (dipilih dari {len(XGB_GRID)} kombinasi via akurasi VALID): {best_params}")
    log(f"Akurasi valid dengan kombinasi ini: {best_acc:.3f}")
    pred_valid = encoder.inverse_transform(best_clf.predict(X_valid))
    pred_test = encoder.inverse_transform(best_clf.predict(X_test))
    return best_clf, pred_valid, pred_test

In [5]:
def build_dataset(required_cols=None):
    """
    Baca aligned_daily.csv, filter hari yang punya berita (n_news > 0) --
    supaya TF-IDF, LM, model harga-saja, dan model gabungan semua dievaluasi
    di himpunan hari yang SAMA PERSIS.

    required_cols: kolom tambahan yang wajib tidak-NaN (mis. fitur lag harga
    untuk train_baseline_ts.py/train_combined.py). realized_vol selalu wajib
    ada karena itu dasar volatility_class.

    Return: (train, valid, test) -- masing-masing DataFrame kronologis,
    sudah punya kolom volatility_class & lag1_volatility_class.
    """
    df = pd.read_csv(C.ALIGNED_DAILY_CSV)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    # --- fit & terapkan volatility_class DI SELURUH KALENDER HARI KERJA,
    # SEBELUM filter n_news > 0 -- supaya lag1_volatility_class(t) merujuk ke
    # HARI KERJA SEBELUMNYA secara kalender (persis semantik lag1_realized_vol
    # / lag1_log_return di preprocess_kurs.py), BUKAN "hari BERITA sebelumnya".
    # Kalau filter n_news dilakukan DULU baru di-shift(1), shift-nya melompati
    # hari-hari tanpa berita (94 dari 1.202 hari) dan salah menunjuk ke hari
    # yang lebih jauh ke belakang -- bug yang sempat kejadian di sini.
    n_all = len(df)
    n_train_all = int(n_all * TRAIN_FRAC)
    train_vol = df["realized_vol"].iloc[:n_train_all].dropna()
    _, edges = pd.qcut(train_vol, 3, retbins=True, duplicates="drop")
    edges = edges.copy()
    edges[0], edges[-1] = -float("inf"), float("inf")  # jaga2 nilai valid/test di luar rentang train
    labels = VOLATILITY_LABELS[: len(edges) - 1]

    df["volatility_class"] = pd.cut(df["realized_vol"], bins=edges, labels=labels)
    df["lag1_volatility_class"] = df["volatility_class"].shift(1)
    # versi angka (low=0 < medium=1 < high=2) -- biar bisa dipakai XGBoost
    # sebagai FITUR, bukan cuma buat naive persistence. Ini fitur paling kuat
    # yang tadinya cuma "dibocorkan" ke naive, gak pernah dikasih ke XGBoost.
    ordinal_map = {label: i for i, label in enumerate(labels)}
    # astype(float) WAJIB: tanpa ini kolomnya ikut ber-dtype category (warisan
    # pd.cut), dan XGBoost membacanya sebagai kategori TANPA urutan kalau diberi
    # DataFrame, tapi sebagai angka kalau diberi numpy array (train_combined.py)
    # -- fitur yang sama jadi diperlakukan beda antar model.
    df["lag1_volatility_class_enc"] = df["lag1_volatility_class"].map(ordinal_map).astype(float)

    # fitur harga turunan juga dihitung di kalender lengkap (alasan sama dgn lag1 di atas)
    df = add_price_features(df)

    # --- BARU SEKARANG filter ke hari yang punya berita ----------------------
    df = df[df["n_news"] > 0]
    # n_news > 0 dijamin di sini, jadi pembagian aman (tidak ada bagi nol)
    df["lm_positive_mean"] = df["lm_positive_sum"] / df["n_news"]
    df["lm_negative_mean"] = df["lm_negative_sum"] / df["n_news"]
    required_cols = required_cols or []
    df = df.dropna(subset=["realized_vol"] + required_cols).reset_index(drop=True)

    n = len(df)
    n_train = int(n * TRAIN_FRAC)
    n_valid = int(n * VALID_FRAC)
    train = df.iloc[:n_train].reset_index(drop=True)
    valid = df.iloc[n_train : n_train + n_valid].reset_index(drop=True)
    test = df.iloc[n_train + n_valid :].reset_index(drop=True)
    return train, valid, test

In [6]:
def log_split_summary(log, train, valid, test):
    log("Distribusi kelas volatility_class (train, dipakai fit ambang batas):")
    log(train["volatility_class"].value_counts().to_string())
    log()
    log(f"Train: {len(train):,} hari ({train['date'].min().date()} .. {train['date'].max().date()})")
    log(f"Valid: {len(valid):,} hari ({valid['date'].min().date()} .. {valid['date'].max().date()})")
    log(f"Test : {len(test):,} hari ({test['date'].min().date()} .. {test['date'].max().date()})")
    log()

## 2. `train_baseline_ts.py` — Baseline time-series (tanpa teks)

In [7]:
"""
train_baseline_ts.py
=====================
Baseline TIME-SERIES MURNI -- prediksi volatility_class (low/medium/high)
CUMA dari histori nilai tukar (PRICE_FEATURES di dataset_split.py), TANPA
teks berita sama sekali. Ini "Tahap 3": tolok ukur yang harus dikalahkan
model gabungan di train_combined.py (Tahap 4).

Split & label volatility_class dari dataset_split.py (70/15/15 kronologis,
ambang batas low/medium/high di-fit dari train saja -- lihat docstring di
sana buat alasannya).

DUA MODEL:
  1. Naive persistence : prediksi hari t = kelas volatility hari t-1
     (lag1_volatility_class). Baseline paling minimal, wajib dikalahkan.
  2. XGBoost            : PRICE_FEATURES -> klasifikasi 3 kelas, hyperparameter
     di-tuning via akurasi VALID (grid yang sama dgn train_combined.py). Dipilih (bukan ARIMA) karena nanti gampang digabung dengan
     fitur NLP di train_combined.py -- sama-sama tabel fitur numerik.

SEMUA FITUR SUDAH di-shift(1) ATAU LEBIH (preprocess_kurs.py / dataset_split.py) -- tidak ada
fitur di sini yang mengandung informasi hari t itu sendiri (no leakage).
"""

import joblib
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

import config as C
from dataset_split import PRICE_FEATURES, build_dataset, log_split_summary, tune_xgb

LOG_DIR = C.ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = LOG_DIR / "train_baseline_ts_report.txt"

MODEL_DIR = C.ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
def main():
    lines = []

    def log(msg=""):
        print(msg)
        lines.append(str(msg))

    train, valid, test = build_dataset(required_cols=PRICE_FEATURES)
    log_split_summary(log, train, valid, test)

    labels = sorted(train["volatility_class"].unique())
    y_train, y_valid, y_test = train["volatility_class"], valid["volatility_class"], test["volatility_class"]

    # =========================================================================
    # MODEL 1: Naive persistence
    # =========================================================================
    pred_valid_naive = valid["lag1_volatility_class"]
    pred_test_naive = test["lag1_volatility_class"]
    acc_naive_valid = accuracy_score(y_valid, pred_valid_naive)
    acc_naive_test = accuracy_score(y_test, pred_test_naive)

    log("=" * 60)
    log("MODEL 1 -- NAIVE PERSISTENCE (volatility hari t = volatility hari t-1)")
    log("=" * 60)
    log(f"Akurasi naive valid                   : {acc_naive_valid:.3f}")
    log(f"Akurasi naive test                    : {acc_naive_test:.3f}")
    log()
    log("Classification report (TEST set):")
    log(classification_report(y_test, pred_test_naive, zero_division=0))
    log("Confusion matrix TEST (baris=aktual, kolom=prediksi), label urut " + str(labels))
    log(confusion_matrix(y_test, pred_test_naive, labels=labels))
    log()

    # =========================================================================
    # MODEL 2: XGBoost tanpa NLP (fitur harga saja), di-tuning via VALID
    # =========================================================================
    encoder = LabelEncoder().fit(train["volatility_class"])
    y_train_enc = encoder.transform(y_train)

    log("=" * 60)
    log("MODEL 2 -- XGBOOST TANPA NLP (fitur harga saja)")
    log("=" * 60)
    log(f"Fitur dipakai: {PRICE_FEATURES}")
    clf_xgb, pred_valid_xgb, pred_test_xgb = tune_xgb(
        train[PRICE_FEATURES], y_train_enc, valid[PRICE_FEATURES], y_valid,
        test[PRICE_FEATURES], encoder, log,
    )
    acc_xgb_valid = accuracy_score(y_valid, pred_valid_xgb)
    acc_xgb_test = accuracy_score(y_test, pred_test_xgb)
    log(f"Akurasi XGBoost valid                 : {acc_xgb_valid:.3f}")
    log(f"Akurasi XGBoost test                  : {acc_xgb_test:.3f}")
    log()
    log("Classification report (TEST set):")
    log(classification_report(y_test, pred_test_xgb, zero_division=0))
    log("Confusion matrix TEST (baris=aktual, kolom=prediksi), label urut " + str(labels))
    log(confusion_matrix(y_test, pred_test_xgb, labels=labels))
    log()
    log("Feature importance:")
    for f, imp in sorted(zip(PRICE_FEATURES, clf_xgb.feature_importances_), key=lambda x: -x[1]):
        log(f"  {f}: {imp:.3f}")
    log()

    joblib.dump(clf_xgb, MODEL_DIR / "baseline_ts_xgb.pkl")
    joblib.dump(encoder, MODEL_DIR / "baseline_ts_label_encoder.pkl")
    log(f"Tersimpan: {MODEL_DIR / 'baseline_ts_xgb.pkl'}")
    log(f"Tersimpan: {MODEL_DIR / 'baseline_ts_label_encoder.pkl'}")
    log()

    # =========================================================================
    log("=" * 60)
    log("PERBANDINGAN AKURASI TEST (target: volatility_class, tanpa fitur teks)")
    log("=" * 60)
    log(f"  Naive persistence       : {acc_naive_test:.3f}")
    log(f"  XGBoost tanpa NLP       : {acc_xgb_test:.3f}")

    REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
    log(f"\nLaporan tersimpan: {REPORT_PATH}")


if __name__ == "__main__":
    main()

Distribusi kelas volatility_class (train, dipakai fit ambang batas):
volatility_class
high      274
low       251
medium    247

Train: 772 hari (2021-09-16 .. 2025-03-25)
Valid: 165 hari (2025-03-26 .. 2025-12-09)
Test : 167 hari (2025-12-11 .. 2026-09-01)

MODEL 1 -- NAIVE PERSISTENCE (volatility hari t = volatility hari t-1)
Akurasi naive valid                   : 0.727
Akurasi naive test                    : 0.713

Classification report (TEST set):
              precision    recall  f1-score   support

        high       0.75      0.75      0.75        51
         low       0.71      0.73      0.72        51
      medium       0.69      0.68      0.68        65

    accuracy                           0.71       167
   macro avg       0.71      0.72      0.72       167
weighted avg       0.71      0.71      0.71       167

Confusion matrix TEST (baris=aktual, kolom=prediksi), label urut ['high', 'low', 'medium']
[[38  2 11]
 [ 5 37  9]
 [ 8 13 44]]

MODEL 2 -- XGBOOST TANPA NLP (fit

## 3. `train_tfidf.py` — Model teks saja (TF-IDF & LM sentiment)

In [9]:
"""
train_tfidf.py
===============
XGBoost TEKS SAJA -- dua jalur fitur NLP -> volatility_class (low/medium/high),
TANPA fitur harga sama sekali, dievaluasi pakai split kronologis 70/15/15
dari dataset_split.py:
  MODEL A -- TF-IDF (text_norm_concat) + XGBoost
  MODEL B -- Sentiment Lexicon Loughran-McDonald (mean_lm_polarity, dst) + XGBoost

Gunanya: menguji apakah berita SENDIRIAN bisa memprediksi volatility_class.
Dibandingkan dengan XGBoost tanpa NLP (train_baseline_ts.py) dan XGBoost +
TF-IDF + LM (train_combined.py), ini menunjukkan apakah berita berguna
berdiri sendiri atau hanya sebagai TAMBAHAN di atas histori harga. Naive
persistence dilaporkan juga sebagai acuan universal yang sama.

Hari yang dipakai SAMA PERSIS dengan train_combined.py (required_cols sama),
dan XGBoost di-tuning pakai grid yang sama (tune_xgb di dataset_split.py).

TF-IDF DI-FIT DI TRAIN SAJA:
TfidfVectorizer.fit_transform() cuma dipanggil di train -- valid & test
cuma di-.transform() pakai vocabulary yang sudah dipelajari. Kalau di-fit
dari seluruh data, model "mengintip" kosakata dari peristiwa masa depan
(mis. kata yang cuma muncul di 2026) buat memprediksi masa lalu.
"""

import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

import config as C
from dataset_split import LM_FEATURES, PRICE_FEATURES, build_dataset, log_split_summary, tune_xgb

MIN_DF = 5
MAX_DF = 0.8
NGRAM_RANGE = (1, 2)
MAX_FEATURES = 5000

LOG_DIR = C.ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = LOG_DIR / "train_tfidf_report.txt"

MODEL_DIR = C.ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
def report_block(log, y_valid, pred_valid, y_test, pred_test, labels):
    log(f"Akurasi valid : {accuracy_score(y_valid, pred_valid):.3f}")
    log(f"Akurasi test  : {accuracy_score(y_test, pred_test):.3f}")
    log()
    log("Classification report (TEST set):")
    log(classification_report(y_test, pred_test, zero_division=0))
    log("Confusion matrix TEST (baris=aktual, kolom=prediksi), label urut " + str(labels))
    log(confusion_matrix(y_test, pred_test, labels=labels))
    log()

In [11]:
def main():
    lines = []

    def log(msg=""):
        print(msg)
        lines.append(str(msg))

    # required_cols sama dengan train_combined.py -> himpunan hari identik
    train, valid, test = build_dataset(required_cols=PRICE_FEATURES + LM_FEATURES)
    log_split_summary(log, train, valid, test)

    labels = sorted(train["volatility_class"].unique())
    y_train, y_valid, y_test = train["volatility_class"], valid["volatility_class"], test["volatility_class"]
    encoder = LabelEncoder().fit(y_train)
    y_train_enc = encoder.transform(y_train)

    # acuan universal: sama dipakai train_baseline_ts.py & train_combined.py
    naive_acc_test = accuracy_score(y_test, test["lag1_volatility_class"])

    # =========================================================================
    # MODEL A: TF-IDF + XGBoost
    # =========================================================================
    vectorizer = TfidfVectorizer(
        min_df=MIN_DF, max_df=MAX_DF, ngram_range=NGRAM_RANGE,
        max_features=MAX_FEATURES,
    )
    X_train_tfidf = vectorizer.fit_transform(train["text_norm_concat"].fillna(""))
    X_valid_tfidf = vectorizer.transform(valid["text_norm_concat"].fillna(""))
    X_test_tfidf = vectorizer.transform(test["text_norm_concat"].fillna(""))
    log(f"Ukuran vocabulary TF-IDF (fit dari train saja): {len(vectorizer.vocabulary_):,}")
    log()

    log("=" * 60)
    log("MODEL A -- TF-IDF + XGBOOST (teks saja) -> volatility_class")
    log("=" * 60)
    clf_tfidf_xgb, pred_valid_tfidf_xgb, pred_test_tfidf_xgb = tune_xgb(
        X_train_tfidf, y_train_enc, X_valid_tfidf, y_valid, X_test_tfidf, encoder, log,
    )
    report_block(log, y_valid, pred_valid_tfidf_xgb, y_test, pred_test_tfidf_xgb, labels)

    feature_names = np.array(vectorizer.get_feature_names_out())
    top_idx = np.argsort(clf_tfidf_xgb.feature_importances_)[-20:][::-1]
    log("Term TF-IDF paling berpengaruh (feature importance XGBoost):")
    log("  " + ", ".join(feature_names[top_idx]))
    log()

    joblib.dump(vectorizer, MODEL_DIR / "tfidf_vectorizer.pkl")
    joblib.dump(clf_tfidf_xgb, MODEL_DIR / "tfidf_xgb.pkl")
    log(f"Tersimpan: {MODEL_DIR / 'tfidf_vectorizer.pkl'}, tfidf_xgb.pkl")
    log()

    # =========================================================================
    # MODEL B: Sentiment Lexicon (Loughran-McDonald) + XGBoost
    # =========================================================================
    # XGBoost berbasis pohon -- gak butuh scaling, pakai nilai mentah
    X_train_lm = train[LM_FEATURES].fillna(0)
    X_valid_lm = valid[LM_FEATURES].fillna(0)
    X_test_lm = test[LM_FEATURES].fillna(0)
    log("=" * 60)
    log("MODEL B -- SENTIMENT LEXICON (LM) + XGBOOST (teks saja) -> volatility_class")
    log("=" * 60)
    log(f"Fitur dipakai: {LM_FEATURES}")
    clf_lm_xgb, pred_valid_lm_xgb, pred_test_lm_xgb = tune_xgb(
        X_train_lm, y_train_enc, X_valid_lm, y_valid, X_test_lm, encoder, log,
    )
    report_block(log, y_valid, pred_valid_lm_xgb, y_test, pred_test_lm_xgb, labels)

    joblib.dump(clf_lm_xgb, MODEL_DIR / "lm_xgb.pkl")
    log(f"Tersimpan: {MODEL_DIR / 'lm_xgb.pkl'}")
    log()

    # =========================================================================
    log("=" * 60)
    log("PERBANDINGAN AKURASI TEST (target: volatility_class, teks saja)")
    log("=" * 60)
    log(f"  Naive persistence (acuan harga) : {naive_acc_test:.3f}")
    log(f"  TF-IDF + XGBoost                : {accuracy_score(y_test, pred_test_tfidf_xgb):.3f}")
    log(f"  LM     + XGBoost                : {accuracy_score(y_test, pred_test_lm_xgb):.3f}")

    REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
    log(f"\nLaporan tersimpan: {REPORT_PATH}")


if __name__ == "__main__":
    main()

Distribusi kelas volatility_class (train, dipakai fit ambang batas):
volatility_class
high      274
low       251
medium    247

Train: 772 hari (2021-09-16 .. 2025-03-25)
Valid: 165 hari (2025-03-26 .. 2025-12-09)
Test : 167 hari (2025-12-11 .. 2026-09-01)

Ukuran vocabulary TF-IDF (fit dari train saja): 5,000

MODEL A -- TF-IDF + XGBOOST (teks saja) -> volatility_class
Hyperparameter terbaik (dipilih dari 27 kombinasi via akurasi VALID): {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}
Akurasi valid dengan kombinasi ini: 0.388
Akurasi valid : 0.388
Akurasi test  : 0.287

Classification report (TEST set):
              precision    recall  f1-score   support

        high       0.28      0.76      0.41        51
         low       0.00      0.00      0.00        51
      medium       0.38      0.14      0.20        65

    accuracy                           0.29       167
   macro avg       0.22      0.30      0.20       167
weighted avg       0.23      0.29      0.20      

## 4. `train_combined.py` — Model gabungan harga + TF-IDF + LM

In [12]:
"""
train_combined.py
==================
MODEL GABUNGAN -- satu XGBoost yang makan SEMUA fitur sekaligus: harga
historis (PRICE_FEATURES di dataset_split.py) + TF-IDF (dikompres) + Loughran-
McDonald sentiment. Ini produk akhirnya -- bukan perbandingan "pilih salah
satu", tapi integrasi ketiganya jadi satu prediktor volatility_class.

Perbandingan utama: (1) naive persistence, (2) XGBoost tanpa NLP (harga
saja), (3) XGBoost + TF-IDF + LM. Selisih (3) vs (2) = kontribusi berita.
Semua XGBoost di-tuning pakai grid yang sama (XGB_GRID di dataset_split.py)
supaya adil. Harga + LM tanpa TF-IDF tetap dilaporkan sebagai ablasi.

KENAPA TF-IDF DIKOMPRES (TruncatedSVD) SEBELUM DIGABUNG:
--------------------------------------------------------------------------
TF-IDF menghasilkan 5.000 kolom (satu per kata/frasa), sedangkan fitur
harga+LM cuma belasan kolom. Kalau digabung mentah-mentah, split tree XGBoost
nyaris selalu jatuh ke salah satu dari 5.000 kolom TF-IDF (probabilitas
menang cuma dari jumlah kolom), dan kolom harga+LM nyaris gak pernah
kepakai walau sinyalnya kuat. TruncatedSVD meringkas 5.000 kolom TF-IDF
jadi N_SVD_COMPONENTS "sumbu topik" utama (kombinasi linear kata-kata yang
sering muncul bareng) -- masih representasi TF-IDF, cuma diringkas supaya
adil headcount-nya lawan fitur harga+LM.

TF-IDF & SVD DI-FIT DI TRAIN SAJA (pola yang sama kayak train_tfidf.py):
fit_transform() di train, transform() doang di valid/test.
"""

import joblib
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import numpy as np
from scipy.stats import binomtest

import config as C
from dataset_split import LM_FEATURES, PRICE_FEATURES, build_dataset, log_split_summary, tune_xgb

MIN_DF = 5
MAX_DF = 0.8
NGRAM_RANGE = (1, 2)
MAX_FEATURES = 5000
N_SVD_COMPONENTS = 20

LOG_DIR = C.ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = LOG_DIR / "train_combined_report.txt"

MODEL_DIR = C.ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [13]:
def mcnemar(y_true, pred_a, pred_b):
    """
    Uji McNemar exact: apakah model A dan B beda akurasi secara signifikan di
    hari-hari yang SAMA. Cuma hari yang hasilnya BEDA (A benar & B salah, atau
    sebaliknya) yang dihitung; di bawah H0 keduanya sama mungkin (binomial p=0,5).
    """
    y_true, pred_a, pred_b = map(np.asarray, (y_true, pred_a, pred_b))
    a_only = int(((pred_a == y_true) & (pred_b != y_true)).sum())
    b_only = int(((pred_a != y_true) & (pred_b == y_true)).sum())
    p = binomtest(a_only, a_only + b_only).pvalue if a_only + b_only else 1.0
    return a_only, b_only, p

In [14]:
def report_block(log, y_valid, pred_valid, y_test, pred_test, labels):
    log(f"Akurasi valid : {accuracy_score(y_valid, pred_valid):.3f}")
    log(f"Akurasi test  : {accuracy_score(y_test, pred_test):.3f}")
    log()
    log("Classification report (TEST set):")
    log(classification_report(y_test, pred_test, zero_division=0))
    log("Confusion matrix TEST (baris=aktual, kolom=prediksi), label urut " + str(labels))
    log(confusion_matrix(y_test, pred_test, labels=labels))
    log()

In [15]:
def main():
    lines = []

    def log(msg=""):
        print(msg)
        lines.append(str(msg))

    train, valid, test = build_dataset(required_cols=PRICE_FEATURES + LM_FEATURES)
    log_split_summary(log, train, valid, test)

    labels = sorted(train["volatility_class"].unique())
    y_train, y_valid, y_test = train["volatility_class"], valid["volatility_class"], test["volatility_class"]
    encoder = LabelEncoder().fit(y_train)
    y_train_enc = encoder.transform(y_train)

    naive_acc = accuracy_score(y_test, test["lag1_volatility_class"])

    # --- fitur TF-IDF, dikompres jadi N_SVD_COMPONENTS kolom, fit di train saja
    vectorizer = TfidfVectorizer(
        min_df=MIN_DF, max_df=MAX_DF, ngram_range=NGRAM_RANGE, max_features=MAX_FEATURES,
    )
    X_train_tfidf = vectorizer.fit_transform(train["text_norm_concat"].fillna(""))
    X_valid_tfidf = vectorizer.transform(valid["text_norm_concat"].fillna(""))
    X_test_tfidf = vectorizer.transform(test["text_norm_concat"].fillna(""))

    svd = TruncatedSVD(n_components=N_SVD_COMPONENTS, random_state=42)
    Z_train = svd.fit_transform(X_train_tfidf)
    Z_valid = svd.transform(X_valid_tfidf)
    Z_test = svd.transform(X_test_tfidf)
    svd_cols = [f"tfidf_svd_{i}" for i in range(N_SVD_COMPONENTS)]
    log(f"TF-IDF vocabulary: {len(vectorizer.vocabulary_):,} kata -> dikompres jadi {N_SVD_COMPONENTS} kolom "
        f"(menjelaskan {svd.explained_variance_ratio_.sum()*100:.1f}% varians)")
    log()

    def make_X(part, Z, cols):
        price_lm = part[PRICE_FEATURES + LM_FEATURES].fillna(0).to_numpy()
        return np.hstack([price_lm, Z]), PRICE_FEATURES + LM_FEATURES + cols

    # =========================================================================
    # TAHAPAN A: XGBoost tanpa NLP (harga saja; sama dengan train_baseline_ts.py)
    # =========================================================================
    log("=" * 60)
    log("XGBOOST TANPA NLP -- harga historis saja")
    log("=" * 60)
    clf_price, pred_valid_price, pred_test_price = tune_xgb(
        train[PRICE_FEATURES], y_train_enc, valid[PRICE_FEATURES], y_valid,
        test[PRICE_FEATURES], encoder, log,
    )
    log()
    acc_price_test = accuracy_score(y_test, pred_test_price)

    # =========================================================================
    # TAHAPAN B (ablasi): harga + LM (tanpa TF-IDF)
    # =========================================================================
    feat_b = PRICE_FEATURES + LM_FEATURES
    log("=" * 60)
    log("ABLASI -- harga + LM (tanpa TF-IDF)")
    log("=" * 60)
    clf_b, pred_valid_b, pred_test_b = tune_xgb(
        train[feat_b].fillna(0), y_train_enc, valid[feat_b].fillna(0), y_valid,
        test[feat_b].fillna(0), encoder, log,
    )
    log()
    acc_b_test = accuracy_score(y_test, pred_test_b)

    # =========================================================================
    # MODEL GABUNGAN AKHIR: harga + LM + TF-IDF(SVD) -- SEMUA fitur sekaligus
    # =========================================================================
    X_train_full, feat_names = make_X(train, Z_train, svd_cols)
    X_valid_full, _ = make_X(valid, Z_valid, svd_cols)
    X_test_full, _ = make_X(test, Z_test, svd_cols)

    log("=" * 60)
    log("MODEL GABUNGAN AKHIR -- harga historis + TF-IDF(SVD) + LM sentiment")
    log("=" * 60)
    log(f"Fitur dipakai ({len(feat_names)} kolom): {feat_names}")
    log()
    clf_combined, pred_valid_combined, pred_test_combined = tune_xgb(
        X_train_full, y_train_enc, X_valid_full, y_valid, X_test_full, encoder, log,
    )
    acc_combined_test = accuracy_score(y_test, pred_test_combined)
    log()
    report_block(log, y_valid, pred_valid_combined, y_test, pred_test_combined, labels)

    log("Feature importance (top 10):")
    importances = sorted(zip(feat_names, clf_combined.feature_importances_), key=lambda x: -x[1])
    for f, imp in importances[:10]:
        tag = "harga" if f in PRICE_FEATURES else ("LM" if f in LM_FEATURES else "TF-IDF")
        log(f"  [{tag}] {f}: {imp:.3f}")
    log(f"  ... total importance TF-IDF(SVD) : {sum(i for f, i in importances if f in svd_cols):.3f}")
    log(f"  ... total importance harga        : {sum(i for f, i in importances if f in PRICE_FEATURES):.3f}")
    log(f"  ... total importance LM           : {sum(i for f, i in importances if f in LM_FEATURES):.3f}")
    log()

    joblib.dump(vectorizer, MODEL_DIR / "combined_tfidf_vectorizer.pkl")
    joblib.dump(svd, MODEL_DIR / "combined_tfidf_svd.pkl")
    joblib.dump(clf_combined, MODEL_DIR / "combined_xgb.pkl")
    joblib.dump(encoder, MODEL_DIR / "combined_label_encoder.pkl")
    log(f"Tersimpan: combined_tfidf_vectorizer.pkl, combined_tfidf_svd.pkl, combined_xgb.pkl, combined_label_encoder.pkl")
    log()

    # =========================================================================
    log("=" * 60)
    log("PERBANDINGAN AKURASI TEST (target: volatility_class)")
    log("=" * 60)
    log(f"  1. Naive persistence (baseline)                : {naive_acc:.3f}")
    log(f"  2. XGBoost tanpa NLP (harga saja)              : {acc_price_test:.3f}   (selisih vs (1): {acc_price_test-naive_acc:+.3f})")
    log(f"  3. XGBoost + TF-IDF + LM -- MODEL UTAMA        : {acc_combined_test:.3f}   (selisih vs (2): {acc_combined_test-acc_price_test:+.3f})")
    log()
    log(f"  Ablasi: XGBoost harga + LM (tanpa TF-IDF)      : {acc_b_test:.3f}   (selisih vs (2): {acc_b_test-acc_price_test:+.3f})")
    log()
    log("Uji signifikansi McNemar (TEST, alpha=0,05) -- 'A saja benar / B saja benar':")
    naive_pred = test["lag1_volatility_class"].astype(str)
    for name, pa, pb in (
        ("(2) vs (1) naive      ", pred_test_price, naive_pred),
        ("(3) vs (2) tanpa NLP  ", pred_test_combined, pred_test_price),
        ("(3) vs (1) naive      ", pred_test_combined, naive_pred),
    ):
        a_only, b_only, p = mcnemar(y_test, pa, pb)
        log(f"  {name}: {a_only:>3} / {b_only:<3} hari  p = {p:.3f}  {'SIGNIFIKAN' if p < 0.05 else 'tidak signifikan'}")

    REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
    log(f"\nLaporan tersimpan: {REPORT_PATH}")


if __name__ == "__main__":
    main()

Distribusi kelas volatility_class (train, dipakai fit ambang batas):
volatility_class
high      274
low       251
medium    247

Train: 772 hari (2021-09-16 .. 2025-03-25)
Valid: 165 hari (2025-03-26 .. 2025-12-09)
Test : 167 hari (2025-12-11 .. 2026-09-01)

TF-IDF vocabulary: 5,000 kata -> dikompres jadi 20 kolom (menjelaskan 24.6% varians)

XGBOOST TANPA NLP -- harga historis saja
Hyperparameter terbaik (dipilih dari 27 kombinasi via akurasi VALID): {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}
Akurasi valid dengan kombinasi ini: 0.806

ABLASI -- harga + LM (tanpa TF-IDF)
Hyperparameter terbaik (dipilih dari 27 kombinasi via akurasi VALID): {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1}
Akurasi valid dengan kombinasi ini: 0.818

MODEL GABUNGAN AKHIR -- harga historis + TF-IDF(SVD) + LM sentiment
Fitur dipakai (33 kolom): ['lag1_log_return', 'lag2_log_return', 'lag3_log_return', 'lag4_log_return', 'known4_mean', 'known4_std', 'vol_if_r0', 'vol_if_rtyp_pos', 

## 5. Ringkasan Hasil Evaluasi

Test set: 167 hari (2025-12-11 .. 2026-09-01). Angka diambil dari output sel di atas.

| Model | Akurasi valid | Akurasi test | Macro F1 test |
|---|---|---|---|
| TF-IDF + XGBoost (teks saja) | 0.388 | 0.287 | 0.20 |
| LM sentiment + XGBoost (teks saja) | 0.370 | 0.305 | 0.31 |
| Naive persistence | 0.727 | 0.713 | 0.72 |
| XGBoost harga saja | 0.806 | 0.731 | 0.74 |
| XGBoost harga + LM (ablasi) | 0.818 | 0.737 | – |
| **XGBoost harga + TF-IDF(SVD) + LM (model utama)** | 0.812 | **0.766** | **0.77** |

Uji McNemar (test, α = 0,05): harga-saja vs naive p = 0.743; model utama vs harga-saja p = 0.146;
model utama vs naive p = 0.188 — tidak ada selisih yang signifikan secara statistik.

**Interpretasi:**
- Teks berita sendirian tidak memprediksi volatilitas (setara tebak kelas mayoritas).
- Menambahkan fitur berita ke fitur harga menaikkan akurasi test +3,6 poin (0.731 → 0.766), tetapi belum signifikan.
- Feature importance model utama: harga 69%, TF-IDF(SVD) 25,5%, LM 5,5%.